# Video Games Sales Data
This dataset contains records of popular video games in North America, Japan, Europe and other parts of the world. Every video game in this dataset has at least 100k global sales.

Not sure where to begin? Scroll to the bottom to find challenges!

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sales = pd.read_csv("vgsales.csv", index_col=0)
print(sales.shape)
sales.head(100)

(16598, 10)


,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
Rank,,,,,,,,,,
1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37
...,...,...,...,...,...,...,...,...,...,...
96,Crash Bandicoot 2: Cortex Strikes Back,PS,1997.0,Platform,Sony Computer Entertainment,3.78,2.17,1.31,0.31,7.58
97,Super Mario Bros. 2,NES,1988.0,Platform,Nintendo,5.39,1.18,0.70,0.19,7.46
98,Super Smash Bros. for Wii U and 3DS,3DS,2014.0,Fighting,Nintendo,3.24,1.35,2.42,0.43,7.45


[Source](https://www.kaggle.com/gregorut/videogamesales) of dataset.

## Data Dictionary

| Column        | Explanation                                                                   |
| ------------- | ----------------------------------------------------------------------------- |
| Rank          | Ranking of overall sales                                                      |
| Name          | Name of the game                                                              |
| Platform      | Platform of the games release (i.e. PC,PS4, etc.)                             |
| Year          | Year the game was released in                                                 |
| Genre         | Genre of the game                                                             |
| Publisher     | Publisher of the game                                                         |
| NA_Sales      | Number of sales in North America (in millions)                                |
| EU_Sales      | Number of sales in Europe (in millions)                                       |
| JP_Sales      | Number of sales in Japan (in millions)                                        |
| Other_Sales   | Number of sales in other parts of the world (in millions)                     |
| Global_Sales  | Number of total sales (in millions)                                           |

# primary research questions
## 1: Did the latest Super Mario game gather enough sales to warrant making another one? if not, should i risk making another Super Mario game based on the sales of the games before it and count the latest one as a fluke (one-time fail)?

## 2: Should it be a non-platformer spin-off? If so, which genre _should_ the game be?

## 3: when comparing a _non-Mario_ Nintendo game from any genre (that isn't a platformer) to a _Mario game_ from that genre, does the Mario IP increase sales enough to warrant using it? should i use a different Nintendo-owned IP instead, like Nintendo Mii? (Nintendo Mii are used by games like Wii sports, Wii Fit, etc.)

## 4: Does a specific region like _non-platformer_ Super Mario games more than others? if so, how big is the margin?

In [3]:
# Data Cleaning
Nintendo = sales.query('Publisher == "Nintendo"') # games Nintendo published.

Nintendo = Nintendo.drop('Publisher', axis=1) # removes the unnecessary Publisher column since all games inside the Nintendo dataframe are Published by them.

Nintendo['Genre'].replace('Platform', 'Platformer', inplace=True) # rename Platform genre(!) to Platformer to avoid calling it wrong.

Nintendo.rename(columns={'Platform': 'Console'}, inplace=True) # rename the Platform column(!) to Console to avoid confusion with the Platformer game genre.

Nintendo.loc[
    (Nintendo['JP_Sales'] == 0) & (Nintendo['Name'] == "Dance Dance Revolution: Mario Mix"),
    'JP_Sales'
] = 0.05 # changes the sale amount in the JP_Sales column of the game "Dance Dance Revolution: Mario Mix" (index 4223) to match the one in "Dance Dance Revolution: Mario Mix (JP sales)" (this is a diffrent row with the index 12772).

Nintendo = Nintendo.drop(index=12772) # drops the row whose data contains only the sales in JP_Sales for the game "Dance Dance Revolution: Mario Mix".

Nintendo.loc[
    (Nintendo['JP_Sales'] == 0) & (Nintendo['Name'] == "Yoshi Touch & Go"),
    'JP_Sales'
] = 0.22 # changes the sale amount in the JP_Sales column of the game "Yoshi Touch & Go" (index 4924) to match the one in "Yoshi Touch & Go (JP sales)" (this is a diffrent row with the index 7284).

Nintendo = Nintendo.drop(index=7284) # drops the row whose data contains only the sales in JP_Sales for the game "Yoshi Touch & Go".

Mario = Nintendo[
    (
        Nintendo['Name'].str.contains("Mario|Luigi|Peach|Daisy|Yoshi|Toad|Bowser|Wario|Waluigi") # Nintendo games whose title contains popular characters from the Super Mario IP.
        | Nintendo['Name'].str.contains("Smash") # includes the Super Smash Bros. franchise, which uses the Super Mario IP for games belonging to the "Fighting" genre.
    ) 
    & 
    (~Nintendo['Name'].str.contains("Fling")) # removes a non-Mario Nintendo game called "FlingSmash".
]

Mario # Nintendo games that use the Super Mario IP (includes the "Super Smash Bros." franchise, and the "Mario & Sonic at the Olympic Games" game series. does NOT include "Mario & Sonic at the Olympic Games" from 2007 because it was published by Sega, owners of the Sonic IP).

,Name,Console,Year,Genre,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
Rank,,,,,,,,,
2,Super Mario Bros.,NES,1985.0,Platformer,29.08,3.58,6.81,0.77,40.24
3,Mario Kart Wii,Wii,2008.0,Racing,15.85,12.88,3.79,3.31,35.82
7,New Super Mario Bros.,DS,2006.0,Platformer,11.38,9.23,6.50,2.90,30.01
9,New Super Mario Bros. Wii,Wii,2009.0,Platformer,14.59,7.06,4.70,2.26,28.62
12,Mario Kart DS,DS,2005.0,Racing,9.81,7.57,4.13,1.92,23.42
...,...,...,...,...,...,...,...,...,...
11284,Famicom Mini: Mario Bros.,GBA,2004.0,Platformer,0.00,0.00,0.08,0.00,0.08
12125,Mario Tennis,Wii,2010.0,Sports,0.00,0.06,0.00,0.01,0.07
12375,Mario vs. Donkey Kong: Tipping Stars,3DS,2015.0,Puzzle,0.00,0.00,0.06,0.00,0.06


In [10]:
#Exploratory Data Analysis
print(Mario.isna().sum()) # amount of missing data in the Mario dataframe

print('-------------') # used to space results more.
print(Mario.duplicated().any()) # check if any values are duplicated.

print('-------------') # used to space results more.
print(Mario.shape) # show amount of rows and columns.

print('-------------') # used to space results more.
print(Mario.columns) # show all column names.

print('-------------') # used to space results more.
print(Mario.nunique(axis=0)) # number of unique values for each column.

##Mario.describe() (some columns got skipped so i printed them one by one.)
print('-------------') # used to space results more.
print("Mario['Year']")
print(Mario['Year'].describe())

print('-------------') # used to space results more.
print("Mario['NA_Sales']") # name of the column
print(Mario['NA_Sales'].describe())

print('-------------') # used to space results more.
print("Mario['EU_Sales']")
print(Mario['EU_Sales'].describe())

print('-------------') # used to space results more.
print("Mario['JP_Sales']")
print(Mario['JP_Sales'].describe())

print('-------------') # used to space results more.
print("Mario['Other_Sales']")
print(Mario['Other_Sales'].describe())

print('-------------') # used to space results more.
print("Mario['Global_Sales']")
print(Mario['Global_Sales'].describe())

## outliars/extreme values (found using IQR)
print('-------------') # used to space results more.
def find_outliers_iqr(data):
    Q1 = np.percentile(data, 25)
    Q3 = np.percentile(data, 75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data < lower_bound) | (data > upper_bound)]
    return outliers
sales_columns = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']
for col in sales_columns:
    outliers_iqr = find_outliers_iqr(Mario[col])
    print(f"Outliers in {col} using IQR:\n{outliers_iqr}\n-------------")

## outliars/extreme values in Global_Sales
outliers_global = find_outliers_iqr(Mario['Global_Sales'])
print(f"Outliers in Global_Sales:\n{outliers_global}")

Name            0
Console         0
Year            0
Genre           0
NA_Sales        0
EU_Sales        0
JP_Sales        0
Other_Sales     0
Global_Sales    0
dtype: int64
-------------
False
-------------
(136, 9)
-------------
Index(['Name', 'Console', 'Year', 'Genre', 'NA_Sales', 'EU_Sales', 'JP_Sales',
       'Other_Sales', 'Global_Sales'],
      dtype='object')
-------------
Name            122
Console          10
Year             32
Genre             9
NA_Sales        109
EU_Sales         84
JP_Sales         95
Other_Sales      51
Global_Sales    129
dtype: int64
-------------
Mario['Year']
count     136.000000
mean     2003.867647
std         7.775373
min      1983.000000
25%      1999.750000
50%      2004.000000
75%      2010.000000
max      2016.000000
Name: Year, dtype: float64
-------------
Mario['NA_Sales']
count    136.000000
mean       2.255588
std        3.714214
min        0.000000
25%        0.367500
50%        0.940000
75%        2.620000
max       29.080000
Name: 

In [5]:
#scatter plot
sales_columns = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']
long_Mario = pd.melt(
    Mario,
    id_vars=['Global_Sales'],
    value_vars=sales_columns,
    var_name='Region',
    value_name='Sales'
)

sns.scatterplot(
    data=long_Mario,
    x='Sales',
    y='Global_Sales',
    hue='Region'
)

plt.show()

In [4]:
# heatmap
sns.heatmap(Mario.corr(numeric_only=True), annot=True)

In [11]:
# primary research - question 1 result
median_sales = Mario['Global_Sales'].median()

sorted_mario = Mario.sort_values('Year')

latest_game = sorted_mario.iloc[-1] # newest Mario game in the dataframe.
previous_game = sorted_mario.iloc[-2]

avg_sales = Mario['Global_Sales'].mean()

if latest_game['Global_Sales'] >= avg_sales:
    print("The latest Super Mario game performed greater that or equal to the average, another game should be made.") # did the latest Mario game perform well compared to the usual sales of a Mario game?
else:
    if (latest_game['Genre'] != previous_game['Genre']) or (latest_game['Console'] != previous_game['Console']):
        prev_sales = previous_game['Global_Sales'] # if the latest game underperformed, check the if game before it belongs to a different Genre or Platform.
        print("The latest game is likely a fluke. another game could be made") # fluke is calculated by if latest_game and previous_game were in the same genre or platform and latest_game underperformed. in other words: Different Genre or Platform & latest underperforms = likely a fluke. Same Genre & Platform + latest underperforms = likely an actual decline and not a fluke.
        
        print(f"The sales of the latest game ({previous_game['Name']}) on the {previous_game['Console']} console had the {previous_game['Genre']} genre and sold {prev_sales} million.")
    else:
        print("The latest game underperformed, even though the previous game belongs to the same Genre and Platform.")

The latest game is likely a fluke. another game could be made
The sales of the latest game (Mario & Sonic at the Rio 2016 Olympic Games) on the 3DS console had the Action genre and sold 0.46 million.


In [146]:
# primary research - question 2 result
latest_game_name = "Mario & Sonic at the Rio 2016 Olympic Games" # newest Mario game in the dataframe.

latest_game_row = Mario[Mario['Name'] == latest_game_name].iloc[0]

latest_console = latest_game_row['Console'] # which console did the newest Mario game in the dataframe belong to?

latest_game_genre = latest_game_row['Genre'] # which genre did the newest Mario game in the dataframe belong to?

target_genre = 'Platformer' # should the next game be a platformer?

if latest_game_genre != target_genre: # calculate if the next game should be a platformer.
    next_game_genre = target_genre
else:
    next_game_genre = latest_game_genre
print(f"Latest game: {latest_game_name}")
print(f"Genre of latest game: {latest_game_genre}")
print(f"used on the {latest_console} console")
print(f"The genre the next game should be in is a {next_game_genre}")

In [6]:
# primary research - question 3 result
sales_increase_threshold = 0.10 # sets sales increase threshold to 10%

genres = Nintendo['Genre'].unique()  # all unique genres in the Nintendo dataframe

for genre in genres:  # Loop through each genre.
    non_mario_games = Nintendo[
        (Nintendo['Genre'] == genre) & 
        (~Nintendo['Name'].str.contains('Mario'))] # all non-Mario Nintendo games with highest sales in all genres.
    if non_mario_games.empty:
        continue
    
    top_non_mario = non_mario_games.sort_values(by='Global_Sales', ascending=False).iloc[0]
    non_mario_sales = top_non_mario['Global_Sales'] # the top non-Mario game with the highest sales globaly.
    
    # Get Mario Nintendo game with highest sales in this genre
    Mario = Nintendo[
        (Nintendo['Genre'] == genre) & 
        (Nintendo['Name'].str.contains('Mario'))]
    if Mario.empty:
        continue
    
    top_mario = Mario.sort_values(by='Global_Sales', ascending=False).iloc[0]
    mario_sales = top_mario['Global_Sales'] # the top game with the highest sales globaly that uses the Mario IP
    
    sales_increase = (mario_sales - non_mario_sales) / non_mario_sales # do the sales increase?
    
    # should you use the Mario IP
    if sales_increase >= sales_increase_threshold:
        decision = f"In genre '{genre}': Mario IP is better, consider using it."
    else:
        decision = f"In genre '{genre}': Mario IP is not recommended, consider a different Nintendo IP."
    
    # results
    print(f"\nGenre: {genre}") # prints the Genre the code is referring to
    print(f"Non-Mario sales: {non_mario_sales:.2f} million ({top_non_mario['Name']})") # prints the sales of the top non-Mario game in that genre
    print(f"Mario sales: {mario_sales:.2f} million ({top_mario['Name']})") # prints the sales of the top game in that genre that uses the Mario IP
    print(f"Sales increase: {sales_increase*100:.2f}%")
    print(decision)

In [12]:
# primary research - question 4 result

mario_platformer = Mario[Mario['Genre'].str.contains('Platformer', case=False, na=False)]
mario_non_platformer = Mario[~Mario['Genre'].str.contains('Platformer', case=False, na=False)]

# Sum regional sales for non-Platformer Mario games
na_sales_non_platformer = mario_non_platformer['NA_Sales'].sum()
eu_sales_non_platformer = mario_non_platformer['EU_Sales'].sum()
jp_sales_non_platformer = mario_non_platformer['JP_Sales'].sum()
other_sales_non_platformer = mario_non_platformer['Other_Sales'].sum()

# Sum regional sales for Platformer Mario games
na_sales_platformer = mario_platformer['NA_Sales'].sum()
eu_sales_platformer = mario_platformer['EU_Sales'].sum()
jp_sales_platformer = mario_platformer['JP_Sales'].sum()
other_sales_platformer = mario_platformer['Other_Sales'].sum()

# Calculate margins (difference in sales)
margin_na = na_sales_non_platformer - na_sales_platformer
margin_eu = eu_sales_non_platformer - eu_sales_platformer
margin_jp = jp_sales_non_platformer - jp_sales_platformer
margin_other = other_sales_non_platformer - other_sales_platformer

# Print results
print("Regional preferences for non-Platformer Super Mario games vs. Platformer:")
print(f"North America: Non-Platformer: {na_sales_non_platformer:.2f}M sales. Platformer: {na_sales_platformer:.2f}M sales. Margin: {margin_na:.2f}M")
print(f"Europe: Non-Platformer: {eu_sales_non_platformer:.2f}M sales. Platformer: {eu_sales_platformer:.2f}M sales. Margin: {margin_eu:.2f}M")
print(f"Japan: Non-Platformer: {jp_sales_non_platformer:.2f}M sales. Platformer: {jp_sales_platformer:.2f}M sales. Margin: {margin_jp:.2f}M")
print(f"Other regions: Non-Platformer: {other_sales_non_platformer:.2f}M sales. Platformer: {other_sales_platformer:.2f}M sales. Margin: {margin_other:.2f}M")

# Find the maximum sales value among all regions for non-Platformer Mario games
non_platformer_sales = mario_non_platformer[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']]
max_non_platformer_region_sales = non_platformer_sales.max().max()

# Find which region and game has the max value
## locate the row and column of the max sales
max_value = non_platformer_sales.max().max()
max_row_idx, max_col = non_platformer_sales.stack().idxmax()

print('-------------') # used to space results more.
print(f"The biggest amount of regional sales among non-Platformer Mario games is {max_value:.2f}M")

print('-------------') # used to space results more.
print(f"the Region belongs to the {max_col} column")

print('-------------') # used to space results more.
print(f"details in the Mario dataframe:\n{mario_non_platformer.loc[[max_row_idx]]}")
      

Regional preferences for non-Platformer Super Mario games vs. Platformer:
North America: Non-Platformer: 134.77M sales. Platformer: 171.99M sales. Margin: -37.22M
Europe: Non-Platformer: 65.99M sales. Platformer: 68.32M sales. Margin: -2.33M
Japan: Non-Platformer: 70.81M sales. Platformer: 66.09M sales. Margin: 4.72M
Other regions: Non-Platformer: 15.93M sales. Platformer: 15.68M sales. Margin: 0.25M
-------------
The biggest amount of regional sales among non-Platformer Mario games is 15.85M
-------------
the Region belongs to the NA_Sales column
-------------
details in the Mario dataframe:
                Name Console    Year  ... JP_Sales  Other_Sales  Global_Sales
Rank                                  ...                                    
3     Mario Kart Wii     Wii  2008.0  ...     3.79         3.31         35.82

[1 rows x 9 columns]


# Findings and insights/Reporting and recommendations
## primary research question 1
The latest game (Mario & Sonic at the Rio 2016 Olympic Games) is likely a fluke. another game could be made. it was on the 3DS console, had the Action genre and sold 0.46 million sales.

## primary research question 2
The genre the next game should be in is a Platformer

## primary research question 3
In genre 'Sports': Mario IP is not recommended, consider a different Nintendo IP.

In genre 'Platformer': Mario IP is better, consider using it.

In genre 'Racing': Mario IP is better, consider using it.

In genre 'Role-Playing': Mario IP is not recommended, consider a different Nintendo IP.

In genre 'Puzzle': Mario IP is not recommended, consider a different Nintendo IP.

In genre 'Misc': Mario IP is not recommended, consider a different Nintendo IP.

In genre 'Action': Mario IP is not recommended, consider a different Nintendo IP.

In genre 'Adventure': Mario IP is better, consider using it.

## primary research question 4
The biggest amount of regional sales among non-Platformer Mario games belongs to the NA_Sales column
with 15.85M sales and -37.22M Margin compared to the sales of Platformer Mario games in the region.

## Don't know where to start?

**Challenges are brief tasks designed to help you practice specific skills:**

- 🗺️ **Explore**: Which of the three seventh generation consoles (Xbox 360, Playstation 3, and Nintendo Wii) had the highest total sales globally?
- 📊 **Visualize**: Create a plot visualizing the average sales for games in the most popular three genres. Differentiate between NA, EU, and global sales.
- 🔎 **Analyze**: Are some genres significantly more likely to perform better or worse in Japan than others? If so, which ones?

**Scenarios are broader questions to help you develop an end-to-end project for your portfolio:**

You are working as a data analyst for a video game retailer based in Japan. The retailer typically orders games based on sales in North America and Europe, as the games are often released later in Japan. However, they have found that North American and European sales are not always a perfect predictor of how a game will sell in Japan.

Your manager has asked you to develop a model that can predict the sales in Japan using sales in North America and Europe and other attributes such as the name of the game, the platform, the genre, and the publisher.

You will need to prepare a report that is accessible to a broad audience. It should outline your motivation, steps, findings, and conclusions.